# NCKH FL + DP — notebook mẫu chạy trên Kaggle

1. **Add Input** → tìm dataset *IBM Transactions for Anti Money Laundering (AML)* (tác giả `ealtman2019`) → Add.
2. Settings → **Internet: On** (để clone repo). GPU chỉ cần khi chạy GNN.
3. Run All. Kết quả nằm ở `/kaggle/working/NCKH2026/results/` và được nén thành `/kaggle/working/ket_qua.zip` (tab Output).

Mỗi bước B1–B6 nên là một notebook riêng, sao từ notebook này, chỉ đổi ô **Chạy thí nghiệm**. Quy tắc: `docs/KHUNG_CHOT_BUOC.md`.

In [ ]:
# Đổi NHANH nếu muốn chạy code của nhánh khác (vd. 'em2/lightgbm')
NHANH = 'main'

!rm -rf /kaggle/working/NCKH2026
!git clone -q --depth 1 -b {NHANH} https://github.com/Hip168/NCKH2026.git /kaggle/working/NCKH2026
%cd /kaggle/working/NCKH2026
!git log --oneline -1

In [ ]:
# Kaggle đã có sẵn numpy, pandas, scikit-learn, torch, lightgbm. Chỉ cài phần còn thiếu,
# KHÔNG cài lại cả requirements.txt (sẽ đè phiên bản torch/numpy của Kaggle).
!pip install -q opacus==1.6.0 torch-geometric pyarrow
!python -m pytest -q tests -p no:cacheprovider --ignore=tests/test_consumer.py   # test_consumer cần Postgres/Kafka, bỏ qua trên Kaggle

In [ ]:
import glob
RAW = sorted(glob.glob('/kaggle/input/**/HI-Small_Trans.csv', recursive=True))[0]
ACC = sorted(glob.glob('/kaggle/input/**/HI-Small_accounts.csv', recursive=True))[0]
print(RAW, ACC, sep='\n')

In [ ]:
# Tính đặc trưng một lần (~2–3 phút, RAM ~2,5 GB)
!python scripts/06_ibm_features.py --input {RAW} --view bank

In [ ]:
# B0: đặc trưng tính trên Kaggle phải khớp cách chia đã chốt (3 dòng KHỚP). Không khớp thì DỪNG, báo Đạo.
!python scripts/b0_chia_co_dinh.py --kiem-tra

In [ ]:
# ===== Chạy thí nghiệm (đổi ô này cho từng bước) =====
# Ví dụ: gộp chung đủ dữ liệu, fold 0, để thử máy (~5 phút)
!python scripts/12_ibm_cach1_lo_nho.py --folds 0 --batch-size 512 --lr 0.001 --max-epochs 300 --patience 30 --tag thu_kaggle

In [ ]:
# Nén kết quả để tải về (tab Output → ket_qua.zip). data/processed không nén vì nặng và tính lại được.
!cd /kaggle/working/NCKH2026 && zip -qr /kaggle/working/ket_qua.zip results -x 'results/_luu_tru/*' && ls -lh /kaggle/working/ket_qua.zip